# 05 — Per-Sector Tuning

**Notebook 5 of the *Developer Guide to Disciplined Trading* series.**

> Prerequisites: [`01`](./01-foundations-techtrade-and-analysis.ipynb), [`02`](./02-morning-scan.ipynb), [`03`](./03-single-position-deep-dive.ipynb), [`04`](./04-validation-gate.ipynb).

> **Soft dependencies**: this notebook needs BOTH `[tuneta]` AND `[validation]` extras (tune calls validate internally to gate the candidate):
> ```
> pip install 'openbb-techtrade[tuneta,validation]'
> ```
> Without either, `obb.techtrade.tune(...)` raises `TechtradeDependencyError` with a copy-pasteable hint.

---

## The question this notebook answers

> *"The default indicator periods (RSI 14, MACD 12/26/9, etc.) are conventional but arbitrary. **For my sector, would different periods give a more robust signal?**"*

`obb.techtrade.tune(segment, ...)` is the answer. It:

1. Pools the OHLCV of every symbol in a GICS sector into one `(date, symbol)`-indexed DataFrame.
2. Runs **`tuneta`** (a distance-correlation + Optuna optimizer) over 8 period knobs to propose new periods.
3. Builds a sample `TradePlan` for the sector's benchmark ETF using the candidate periods.
4. Calls **`obb.techtrade.validate(plan)`** to get a verdict on the candidate.
5. **ONLY IF** `verdict == "robust"` AND the candidate is genuinely different from `DEFAULT_CONFIG`, persists it to `~/.openbb_platform/techtrade_tuned.json`.
6. Returns a `TuningReport` with `persisted: bool`, `reason: str`, and the candidate config.

Once persisted, **every subsequent `scan` / `plan` / `signals` call for a symbol in that sector auto-loads the tuned periods** transparently — no caller change required. That's the L9 auto-load wiring from #83.

## What Alex takes away

- A tuned `IndicatorConfig` for one sector + an understanding of which periods changed and why.
- A sense for how often tuned periods actually pass the gate (spoiler: under strict thresholds, most don't — which is the point).
- The mechanics of the persist-and-auto-load chain: how a `tune` call yesterday changes what `scan` returns today.
- The discipline of treating tuneta as a **proposer**, not an oracle — every proposal must clear the same `validate` gate from notebook 04.

## Wall-clock

**5-20 minutes per `tune(segment)` call**. tuneta itself runs ~100 Optuna trials per indicator; then validate runs WFO folds on the candidate. This is the slowest notebook in the series.

## 1. Soft-dep probe + setup

In [ ]:
import importlib.util

TUNETA_AVAILABLE = importlib.util.find_spec("tuneta") is not None
BACKTEST_AVAILABLE = importlib.util.find_spec("openbb_backtest") is not None

print(f"[tuneta]     extra installed:  {TUNETA_AVAILABLE}")
print(f"[validation] extra installed:  {BACKTEST_AVAILABLE}")

if not (TUNETA_AVAILABLE and BACKTEST_AVAILABLE):
    print("\nThis notebook needs BOTH. Install with:")
    print("    pip install 'openbb-techtrade[tuneta,validation]'")
    print("\nCells below will SKIP cleanly until both are present.")

In [ ]:
from openbb import obb
from pathlib import Path

SEGMENT = "Information Technology"  # pick any GICS sector; IT has a large, liquid universe

TUNED_PATH = Path.home() / ".openbb_platform" / "techtrade_tuned.json"
print(f"Studying tune for segment: {SEGMENT}")
print(f"Tuned defaults will be persisted to: {TUNED_PATH}")

## 2. The DEFAULT_CONFIG baseline

Before we tune, look at what we're tuning *away from*. The engine ships with a frozen `IndicatorConfig` containing the conventional periods (RSI 14, MACD 12/26/9, EMA 20/50, ATR 14, etc.). tuneta proposes deltas from this baseline.

In [ ]:
from dataclasses import asdict
from openbb_techtrade.engine.indicators import DEFAULT_CONFIG

import pandas as pd
defaults_df = pd.DataFrame([{"knob": k, "default": v} for k, v in asdict(DEFAULT_CONFIG).items()])
defaults_df

## 3. Which knobs are actually tunable?

tuneta operates on **one period per indicator**. The KNOB_TABLE in `tuning/tuneta_adapter.py` (L6 of the design) ships **8 period knobs** that map cleanly to tuneta's `(low, high)` range syntax:

In [ ]:
if TUNETA_AVAILABLE and BACKTEST_AVAILABLE:
    from openbb_techtrade.tuning.tuneta_adapter import KNOB_TABLE
    knob_df = pd.DataFrame([
        {"field": field, "tuneta_indicator": indicator, "range_low": rng[0], "range_high": rng[1]}
        for field, indicator, rng in KNOB_TABLE
    ])
    print(f"{len(knob_df)} tunable period knobs:")
    knob_df
else:
    print("[tuneta] not installed; cannot show KNOB_TABLE. Install and rerun.")
    knob_df = None

### Why these 8 and not the other 7?

Per the design (issue #83, decision L6), the other 7 `IndicatorConfig` fields are excluded because:

- **Stoch (`stoch_k`, `stoch_d`, `stoch_smooth_k`)** — needs 3 coherent periods together; tuneta's one-knob-at-a-time would propose incoherent combos.
- **Bollinger (`bb_length`, `bb_std`)** — `bb_std` is a shape multiplier, not a period; tuneta has no notion of it.
- **Keltner (`kc_length`, `kc_scalar`)** — same as Bollinger.

These 7 stay at PRD defaults under any tune. Only the 8 above will be different in a tuned config vs `DEFAULT_CONFIG`.

## 4. The current state of `techtrade_tuned.json`

Before tuning, check whether anything is already persisted. The file may or may not exist; if it does, it carries entries from previous tune runs.

In [ ]:
import json

if not TUNED_PATH.exists():
    print("No tuned file yet — every sector currently uses DEFAULT_CONFIG.")
else:
    with open(TUNED_PATH) as fh:
        existing = json.load(fh)
    print(f"Existing tuned file: schema_version={existing.get('schema_version')}")
    print(f"Tuned segments ({len(existing.get('segments', {}))}):")
    for seg, entry in existing.get("segments", {}).items():
        verdict = entry.get("meta", {}).get("verdict", "?")
        tuned_at = entry.get("meta", {}).get("tuned_at", "?")
        print(f"  - {seg:<25} verdict={verdict}  tuned_at={tuned_at}")

## 5. Run `obb.techtrade.tune(segment)` — the headline call

This is the slow cell. The default budget is `trials=100, early_stop=20` per indicator (8 indicators), then a full WFO validate on the candidate.

**Wall-clock: 5-20 minutes.** Go make coffee.

If you want to feel the surface faster, override the budget below (`trials=20, early_stop=5` finishes in ~2-5 min but produces noisier candidates).

In [ ]:
if not (TUNETA_AVAILABLE and BACKTEST_AVAILABLE):
    print("Skipping tune call — need both [tuneta] and [validation].")
    tune_report = None
else:
    import asyncio

    print(f"Running tune({SEGMENT!r}) ... patience ...")
    tune_result = asyncio.run(obb.techtrade.tune(
        segment=SEGMENT,
        # Budget knobs — uncomment to make the cell faster:
        # trials=20,
        # early_stop=5,
    ))
    tune_report = tune_result.results
    print(f"\nDone in {tune_report.fit_seconds:.1f}s of tuneta + {len(tune_report.validation.folds) if tune_report.validation else 0} validate folds.")
    print(f"\nOutcome: persisted={tune_report.persisted}  reason={tune_report.reason!r}")

## 6. Read the `TuningReport`

Whether persisted or not, the report carries everything Alex needs to understand the outcome.

In [ ]:
if tune_report is None:
    print("No report (tune was skipped).")
else:
    print(f"--- TuningReport for {tune_report.segment} ---\n")
    print(f"  segment:         {tune_report.segment}")
    print(f"  as_of:           {tune_report.as_of}")
    print(f"  tuneta version:  {tune_report.tuneta_version}")
    print(f"  trials:          {tune_report.trials}")
    print(f"  early_stop:      {tune_report.early_stop}")
    print(f"  fit_seconds:     {tune_report.fit_seconds:.1f}")
    print(f"  persisted:       {tune_report.persisted}")
    print(f"  reason:          {tune_report.reason}")
    if tune_report.validation is not None:
        v = tune_report.validation
        print(f"\n  Validation:")
        print(f"    verdict:               {v.verdict.upper()}")
        print(f"    PBO:                   {v.pbo:.4f}")
        print(f"    Deflated Sharpe:       {v.deflated_sharpe:.4f}")
        print(f"    OOS Sharpe:            {v.oos_metrics.sharpe:.4f}")
        print(f"    Folds:                 {len(v.folds)}")

### Interpreting the outcome

| Outcome | Reason field | What happened | What Alex does |
|---|---|---|---|
| `persisted=True` | `verdict=robust` | Tuneta found new periods that passed the strict gate. The file was updated. | Re-run `scan` / `plan` / `signals` for this sector — output will reflect the tuned periods automatically. |
| `persisted=False` | `verdict=fragile (pbo=..., dsr=...)` | Periods proposed but not statistically robust. Not persisted. | Stay on `DEFAULT_CONFIG`. Try a different sector or a longer horizon. |
| `persisted=False` | `verdict=overfit (pbo=..., dsr=...)` | Periods proposed but **actively bad** out-of-sample. Strong signal to walk away. | Definitely stay on `DEFAULT_CONFIG`. Don't tune again with the same settings. |
| `persisted=False` | `no change from defaults` | Tuneta proposed periods identical to `DEFAULT_CONFIG`. Q-F guard #3 prevents persisting a no-op. | The PRD defaults already fit this sector well. No action needed. |

Under the strict thresholds (PBO < 0.2, DSR > 0.95) the **majority of tuneta candidates fail the gate**. That's the design intent — it's a *conservative* tuner. A `persisted=True` outcome is meaningful precisely because it's hard to get.

## 7. What did tuneta actually propose?

The `candidate` field is the `IndicatorConfig` tuneta produced. Compare it to `DEFAULT_CONFIG` to see which knobs moved.

In [ ]:
if tune_report is None or tune_report.candidate is None:
    print("No candidate to show.")
else:
    candidate = tune_report.candidate
    default = DEFAULT_CONFIG
    rows = []
    for k in asdict(default):
        d = getattr(default, k)
        c = getattr(candidate, k)
        rows.append({
            "knob": k,
            "default": d,
            "candidate": c,
            "delta": c - d if isinstance(d, (int, float)) and isinstance(c, (int, float)) else None,
            "changed": (c != d),
        })
    df = pd.DataFrame(rows)
    changed = df[df["changed"]]
    if changed.empty:
        print("Tuneta proposed IDENTICAL periods to DEFAULT_CONFIG — no knob moved.")
    else:
        print(f"Tuneta moved {len(changed)} / {len(df)} knobs:\n")
    df

## 8. Verify the persist-and-auto-load chain end-to-end

**The most important behavior in this notebook.** If `persisted=True`, then a fresh call to `obb.techtrade.signals(symbol_in_sector)` should now use the tuned periods automatically — no caller change. The auto-load is in `engine/indicators.py:build_indicator_panel`, which consults `lookup_tuned_for_symbol(symbol)` whenever `config=None`.

Demonstrate by comparing the indicator panel for an IT-sector symbol BEFORE and AFTER tuning. If the periods changed, the indicator values change too.

In [ ]:
if tune_report is None or not tune_report.persisted:
    print("Skip: nothing was persisted, so the auto-load path falls through to DEFAULT_CONFIG (no change to observe).")
else:
    # Confirm the tuned entry is on disk.
    from openbb_techtrade.tuning.tuned_defaults import lookup_tuned_for_symbol, _clear_cache
    _clear_cache()  # belt-and-braces; the mtime cache should invalidate on its own

    # NVDA is in IT. After persist, lookup should return the tuned IndicatorConfig.
    looked_up = lookup_tuned_for_symbol("NVDA")
    if looked_up is None:
        print("⚠ Unexpected: persist=True but lookup returned None. Check the segment resolver.")
    else:
        print(f"Auto-load works! lookup_tuned_for_symbol('NVDA') returns a non-None IndicatorConfig.")
        print(f"  RSI period (tuned):   {looked_up.rsi_length}")
        print(f"  RSI period (default): {DEFAULT_CONFIG.rsi_length}")
        same = looked_up.rsi_length == DEFAULT_CONFIG.rsi_length
        print(f"  Same as default?      {same}")

## 9. Bypassing the auto-load

Sometimes you want the BASELINE behavior even when a tuned config is persisted (e.g. running golden tests, comparing against a reference panel, etc.). Pass an explicit `config=DEFAULT_CONFIG` to `build_indicator_panel`:

In [ ]:
# This is the internal-API path, exposed for tests + advanced users.
# Most consumers never call build_indicator_panel directly — they use
# obb.techtrade.signals/plan/scan which delegate via the auto-load.
from openbb_techtrade.engine.indicators import build_indicator_panel, DEFAULT_CONFIG
import inspect

sig = inspect.signature(build_indicator_panel)
print("build_indicator_panel signature:")
for pname, p in sig.parameters.items():
    default = "" if p.default is inspect.Parameter.empty else f" = {p.default!r}"
    print(f"  {pname}: {p.annotation}{default}")

### The signature contract

- `config=None` (default) → auto-load from `~/.openbb_platform/techtrade_tuned.json` by `symbol → segment`, falling back to `DEFAULT_CONFIG` if no robust entry exists.
- `config=DEFAULT_CONFIG` (explicit) → bypass the auto-load entirely. **Caller intent wins.** Use this in tests where you need bit-for-bit reproducibility against a fixed reference.
- `config=<custom IndicatorConfig>` → use the supplied config. Bypass everything.

## 10. Tuning multiple sectors (the patient path)

Alex wants tuned periods for ALL 11 sectors. The `tune` command is single-segment by design (L7) — caller loops. Below is the disciplined batch loop. **Don't run it in this notebook unless you have ~1-2 hours.** Each call is 5-20 min; 11 sectors × ~10 min = ~110 min total.

In [ ]:
# Template — uncomment to run the whole loop. Will take ~1-2 hours.
# Each iteration's persist outcome accumulates in techtrade_tuned.json without
# disturbing the others.

BATCH_LOOP_TEMPLATE = '''
import asyncio
from openbb import obb

GICS_SECTORS = [s.segment for s in obb.techtrade.segments().results]
results = {}

for sector in GICS_SECTORS:
    print(f"--- Tuning {sector} ---")
    try:
        rep = asyncio.run(obb.techtrade.tune(segment=sector)).results
        results[sector] = {
            "verdict":   rep.validation.verdict if rep.validation else "?",
            "persisted": rep.persisted,
            "reason":    rep.reason,
        }
        print(f"    -> {rep.reason}  persisted={rep.persisted}")
    except Exception as exc:
        results[sector] = {"error": str(exc)}
        print(f"    !! error: {exc}")

# Summary table at the end:
import pandas as pd
pd.DataFrame.from_dict(results, orient="index")
'''
print(BATCH_LOOP_TEMPLATE)

## 11. The discipline of tuning

tuneta is a **proposer**, not an authority. Every proposal must clear the same `validate` gate from notebook 04 — the gate is the contract that prevents the bias-variance trade from going wrong. Three rules Alex follows:

1. **Never disable the gate.** The temptation to lower `pbo_robust` or `dsr_robust` to make more proposals pass is the first step toward overfitting. The defaults are strict on purpose. If you find yourself wanting to relax them, **stop tuning** instead.
2. **Re-tune on a schedule, not on demand.** Quarterly is reasonable. Re-tuning after every losing trade is reactive — exactly the behavior the gate exists to prevent.
3. **Tuned ≠ Better.** A `persisted=True` outcome means the tuned config is *not statistically worse* than `DEFAULT_CONFIG` over the validation window. It does not promise better forward performance. The verdict is necessary, not sufficient.

## 12. What's next

- **Notebook 06 — Audit and Replay**: the last in the series. Load yesterday's Excel + the validation verdicts that came with it; cross-reference what Alex actually did with what the engine said; write a journal entry. End-of-day discipline.

## Reset / rollback

If you want to undo a tune (e.g. revert to `DEFAULT_CONFIG` for one sector):

```python
import json
from pathlib import Path
TUNED_PATH = Path.home() / ".openbb_platform" / "techtrade_tuned.json"
data = json.loads(TUNED_PATH.read_text())
data["segments"].pop("Information Technology", None)  # name the sector to remove
TUNED_PATH.write_text(json.dumps(data, sort_keys=True, indent=2))
```

Or just delete the file entirely to clear all tuned configs at once.

---

*End of notebook 05.*